# Flow & Engagement — Python Baseline for ludoSpring

**SPDX-License-Identifier: AGPL-3.0-or-later**

Reference implementations for Csikszentmihalyi flow theory and engagement
metrics. These produce the ground truth for ludoSpring's interaction science.

### References
- Csikszentmihalyi, M. (1990). "Flow: The Psychology of Optimal Experience."
- Yannakakis, G.N. & Togelius, J. (2018). "Artificial Intelligence and Games."
- Lazzaro, N. (2004). "Why We Play Games: Four Keys to More Emotion."

In [ ]:
import math

# Flow channel parameters (matching tolerances/interaction.rs)
FLOW_CHANNEL_WIDTH = 0.2
DDA_TARGET_SUCCESS_RATE = 0.65

# Engagement weights (matching tolerances/metrics.rs)
WEIGHT_FLOW = 0.35
WEIGHT_CHALLENGE = 0.25
WEIGHT_AUTONOMY = 0.20
WEIGHT_FEEDBACK = 0.20

In [ ]:
def sigmoid(x):
    """Standard logistic sigmoid."""
    return 1.0 / (1.0 + math.exp(-x))

def flow_state(challenge, skill, channel_width=FLOW_CHANNEL_WIDTH):
    """Determine if player is in flow: |challenge - skill| < channel_width."""
    return abs(challenge - skill) < channel_width

def difficulty_curve(x):
    """DDA sigmoid mapping challenge to success probability."""
    return sigmoid(x)

# Flow state examples
print("Flow State Analysis")
print("=" * 50)
cases = [
    (0.5, 0.5, "matched — in flow"),
    (0.8, 0.3, "too hard — anxiety"),
    (0.2, 0.9, "too easy — boredom"),
    (0.6, 0.5, "edge of flow"),
]
for challenge, skill, note in cases:
    in_flow = flow_state(challenge, skill)
    print(f"  C={challenge:.1f} S={skill:.1f} → {'FLOW' if in_flow else 'NOT FLOW'} ({note})")

In [ ]:
def engagement_composite(flow_score, challenge_score, autonomy_score, feedback_score):
    """Weighted engagement composite (dot product with weight vector)."""
    return (
        WEIGHT_FLOW * flow_score +
        WEIGHT_CHALLENGE * challenge_score +
        WEIGHT_AUTONOMY * autonomy_score +
        WEIGHT_FEEDBACK * feedback_score
    )

# Engagement scenarios
print("Engagement Composite Scores")
print("=" * 50)
scenarios = [
    ("Excellent game", 0.9, 0.8, 0.85, 0.9),
    ("Mediocre game", 0.4, 0.5, 0.3, 0.6),
    ("Bad UX, good game", 0.7, 0.8, 0.2, 0.1),
    ("Zero engagement", 0.0, 0.0, 0.0, 0.0),
    ("Perfect scores", 1.0, 1.0, 1.0, 1.0),
]

for name, f, c, a, fb in scenarios:
    score = engagement_composite(f, c, a, fb)
    print(f"  {name:<20} → {score:.4f}")

# Verify weights sum to 1.0
weight_sum = WEIGHT_FLOW + WEIGHT_CHALLENGE + WEIGHT_AUTONOMY + WEIGHT_FEEDBACK
assert abs(weight_sum - 1.0) < 1e-10, f"Weights must sum to 1.0, got {weight_sum}"
print(f"\nWeight sum: {weight_sum} (verified = 1.0)")

In [ ]:
# DDA (Dynamic Difficulty Adjustment) sigmoid curve
print("DDA Sigmoid Curve")
print("=" * 50)
print(f"{'x':>6} {'sigmoid(x)':>12} {'In target zone':>15}")
print("-" * 35)
for x_val in [-3.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]:
    s = sigmoid(x_val)
    in_zone = abs(s - DDA_TARGET_SUCCESS_RATE) < 0.15
    print(f"{x_val:>6.1f} {s:>12.8f} {'YES' if in_zone else '':>15}")

# Validate sigmoid at key points
assert abs(sigmoid(0.0) - 0.5) < 1e-15, "sigmoid(0) must be 0.5"
print("\nsigmoid(0) = 0.5: PASS")

## Validation

These formulas and values are verified in:
- `ludoSpring/barracuda/src/interaction/flow.rs` (flow state)
- `ludoSpring/barracuda/src/metrics/engagement.rs` (composite score)
- `ludoSpring/barracuda/src/interaction/difficulty.rs` (DDA sigmoid)
- Validation scenario: `s_engagement_metrics`

The engagement weights are centralized in `tolerances/metrics.rs`.